# Manufacturer Recalls and Driver Fatalities Analysis

This project explores whether there is a relationship between the number of vehicle recalls issued by manufacturers and the number of fatal accidents involving their vehicles. It was completed as a part of a course project in collaboration with a peer.

The analysis uses aggregated data from the National Highway Traffic Safety Administration (NHTSA) and the Fatality Analysis Reporting System (FARS). The goal is to determine whether recall frequency contributes to fatality rates, or whether other factors—such as vehicle exposure on the road—better explain variation in fatalities across manufacturers.

The notebook focuses primarily on the statistical analysis and modeling portion of the project, including correlation analysis and regression modeling implemented in Python.

Additional project components, including data preprocessing, analysis, and extended interpretation of the results, are discussed in the full final report.

## Import Libraries
The following libraries are used for data manipulation, statistical testing, and regression modeling.

In [1]:
import pandas as pd
from scipy.stats import pearsonr
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score

## Load and Inspect Data

The dataset contains aggregated information about vehicle manufacturers, including:

- Lag Accounted Recalls
- Cars on the Road (COR proxy)
- Total Fatalities
- Manufacturer

This data allows me to analyze relationships between recall frequency, exposure, and fatal accident counts.

In [2]:
df = pd.read_csv("AggregatedRecallandFatalitiesData.csv")
print(df.head())

   Lag Accounted Recalls  DEATHS  Cars on Road  Recall Rate per 100k per COR  \
0               41323755     278       3617543                  1.142316e+06   
1              249501619    1590      20663080                  1.207475e+06   
2               11208366     170       1912154                  5.861644e+05   
3              238777338    1728      24345813                  9.807737e+05   
4              265398232    2545      30131209                  8.808084e+05   

   Fatality Rate per 100k per COR    Manufacturer  
0                        7.684774             BMW  
1                        7.694884        Chrysler  
2                        8.890497         Daimler  
3                        7.097730            Ford  
4                        8.446392  General Motors  


## Pearson Correlation Analysis

Before building predictive models, I test whether there is a statistical relationship between recalls and fatalities.

Pearson's correlation coefficient measures the strength and direction of a linear relationship between two variables.

Values close to:
- **1** indicate a strong positive relationship
- **0** indicate no relationship
- **-1** indicate a strong negative relationship

In [3]:
#creating variables for columns being used in pearson correlation
recalls = df['Lag Accounted Recalls']
deaths = df['DEATHS']

#running pearson's correlation
r,p = pearsonr(recalls, deaths)

print("Pearson correlation coefficient: ", r)
print("p-value: ", p)

Pearson correlation coefficient:  0.9145803157685164
p-value:  0.0005512650293930802


## Multiple Linear Regression

Multiple linear regression examines how multiple predictors influence fatalities.

Predictors used in this model:

- Lag Accounted Recalls
- Cars on the Road (COR proxy)

Target variable:

- Deaths

This model helps determine whether recalls meaningfully contribute to predicting fatalities when controlling for vehicle exposure.

In [4]:
#Setting up our variables
# our multiple independent variables (predictors)
X = df[["Lag Accounted Recalls", "Cars on Road"]]

# dependent variable (only one)
y = df["DEATHS"]

# Initializing the model
MLR_model = LinearRegression()
MLR_model.fit(X, y)

# Results
print("\nMLR Results (X = Recalls, Cars on the Road)     ")

print("y-intercept:", MLR_model.intercept_)
print("Coefficients:", MLR_model.coef_)
print("Recall coefficient:", MLR_model.coef_[0])
print("Cars on the Road coefficient:", MLR_model.coef_[1])

# computing R-squared
MLR_y_pred = MLR_model.predict(X)
print("R-squared score:", r2_score(y, MLR_y_pred))


MLR Results (X = Recalls, Cars on the Road)     
y-intercept: -4.918611207110416
Coefficients: [2.28266879e-08 8.26245232e-05]
Recall coefficient: 2.282668787425568e-08
Cars on the Road coefficient: 8.26245231682947e-05
R-squared score: 0.9311757419824976


## Simple Linear Regression

To compare results, I also build a simple linear regression model using only recall counts as the predictor.

This allows for evaluations of whether recall counts alone provide meaningful predictive power for fatalities.

In [5]:
# SIMPLE LINEAR REGRESSION (Recalls → Deaths only)
SLR_X = df[["Lag Accounted Recalls"]]

# Running the simple model
simple_model = LinearRegression()
simple_model.fit(SLR_X, y)

# Results
print("\nSLR Results (X = Recalls)")

print("y-intercept:", simple_model.intercept_)
print("Recall Coefficient:", simple_model.coef_[0])

SLR_y_pred = simple_model.predict(SLR_X)

print("R-squared Score:", r2_score(y, SLR_y_pred))


SLR Results (X = Recalls)
y-intercept: 100.81680987915627
Recall Coefficient: 7.4920879074303386e-06
R-squared Score: 0.8364571539912395


## Model Prediction Comparison

Finally, run comparison of the predicted fatalities from both models to the actual values.

This allows for evaluation of how well each model approximates the observed fatality counts.

In [6]:
# CALCULATING PREDICTIONS
df["Predicted_Deaths"] = MLR_y_pred

print("\nActual vs. Predicted Fatalities (MLR)")
print(df[["Manufacturer", "DEATHS", "Predicted_Deaths"]])

df["Predicted_Deaths"] = SLR_y_pred

print("\nActual vs. Predicted Fatalities (SLR)")
print(df[["Manufacturer", "DEATHS", "Predicted_Deaths"]])


Actual vs. Predicted Fatalities (MLR)
     Manufacturer  DEATHS  Predicted_Deaths
0             BMW     278        294.922439
1        Chrysler    1590       1708.053817
2         Daimler     170        153.328051
3            Ford    1728       2012.093075
4  General Motors    2545       2490.716328
5           Honda    1490       1201.298165
6        Mercedes     166        311.842606
7          Nissan    1512       1100.389440
8      Volkswagen     345        551.356080

Actual vs. Predicted Fatalities (SLR)
     Manufacturer  DEATHS  Predicted_Deaths
0             BMW     278        410.418015
1        Chrysler    1590       1970.104872
2         Daimler     170        184.790873
3            Ford    1728       1889.757616
4  General Motors    2545       2089.203694
5           Honda    1490       1662.675953
6        Mercedes     166        344.813720
7          Nissan    1512        797.299777
8      Volkswagen     345        474.935478


## Conclusion

This analysis investigated whether vehicle recalls are associated with fatal accident counts across manufacturers.

Pearson correlation suggested a strong statistical relationship between recalls and fatalities. However, regression analysis revealed that recall counts alone contribute very little predictive power once vehicle exposure is considered. The Cars on the Road (COR) proxy emerged as a stronger predictor of fatality counts, suggesting that accident frequency is more strongly related to exposure than recall frequency.

Overall, the results highlight the importance of considering contextual variables when interpreting safety-related metrics. While recalls may signal manufacturing issues, they do not necessarily translate into increased fatality risk at the manufacturer level.
